# NeoRAG — free GPU fine-tuning on Google Colab

Local machine has no CUDA? Run the training step here for free.

1. **Runtime → Change runtime type → T4 GPU (free)**
2. Run the cells below in order.
3. Download the merged model at the end and drop it in your project's `checkpoints/` folder.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/<YOUR_USER>/NeoRAG.git NeoRAG || true
# OR upload the project zip / mount Drive:
# from google.colab import drive; drive.mount('/content/drive')
%cd /content/NeoRAG 2>/dev/null || %cd /content

In [ ]:
!pip install -q -e ".[train]"

## 1. Build the fine-tuning dataset
If you already have `data/finetune/sft.jsonl` (built locally with `python -m neorag.cli build-dataset`), upload it and skip this cell. Otherwise generate it here from `data/corpus/`:

In [ ]:
!python -m neorag.cli build-dataset --per-chunk 1

## 2. Train (LoRA / QLoRA)

In [ ]:
!python -m neorag.cli train

## 3. Merge the adapter into a standalone model

In [ ]:
!python -m neorag.cli merge

In [ ]:
!zip -qr neorag-sft-merged.zip checkpoints/neorag-sft-merged
from google.colab import files
files.download('neorag-sft-merged.zip')

## 4. Use it locally
Unzip into `checkpoints/neorag-sft-merged`, then in `config/neorag.yaml`:

```yaml
llm:
  backend: local
  model_id: checkpoints/neorag-sft-merged
```

Your weak base model is now your own retrieval-grounded RAG agent.